# **Improving Customer Retention through Payment Analytics**

In the **Olist** ecosystem, some customers pay in a single installment, while others use up to 24. The goal is to **analyze if payment installments and payment type (Credit Card vs. Voucher/Boleto) affect Customer Satisfaction (Review Scores) and Delivery Speed**.

## Imports

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import os
from pathlib import Path
import dash
from dash import dcc, html, Input, Output, callback
import plotly.express as px
import plotly.graph_objects as go

## **Download DataSet**

In [2]:
# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

DATA_DIR = Path(path)
print("Dataset downloaded to:", DATA_DIR)
print("\nFiles available:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {f.name}")

Dataset downloaded to: C:\Users\laura\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

Files available:
  olist_customers_dataset.csv
  olist_geolocation_dataset.csv
  olist_order_items_dataset.csv
  olist_order_payments_dataset.csv
  olist_order_reviews_dataset.csv
  olist_orders_dataset.csv
  olist_products_dataset.csv
  olist_sellers_dataset.csv
  product_category_name_translation.csv


## **Data Strategy**

In [3]:
#Load individual tables

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv",
                     parse_dates=[
                         "order_purchase_timestamp",
                         "order_approved_at",
                         "order_delivered_carrier_date",
                         "order_delivered_customer_date",
                         "order_estimated_delivery_date"
                     ])
 
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
 
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])
 
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
 
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv",
                    parse_dates=["shipping_limit_date"])
 
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
 
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
 
print("\nTable shapes:")
for name, df in [("orders", orders), ("payments", payments), ("reviews", reviews),
                 ("customers", customers), ("items", items), ("products", products),
                 ("category_translation", category_translation)]:
    print(f"  {name:25s} {df.shape}")


Table shapes:
  orders                    (99441, 8)
  payments                  (103886, 5)
  reviews                   (99224, 7)
  customers                 (99441, 5)
  items                     (112650, 7)
  products                  (32951, 9)
  category_translation      (71, 2)


### Handling Missing Values

In [4]:
# Orders with missing delivery date are the ones not yet delivered — expected
print(f"\nOrders total:               {len(orders):,}")
print(f"Delivered orders:           {orders['order_delivered_customer_date'].notna().sum():,}")
print(f"Unique customers:           {orders['customer_id'].nunique():,}")
 
# Payments: some orders have multiple payment rows (split payments)
print(f"\nPayment rows:               {len(payments):,}")
print(f"Unique orders in payments:  {payments['order_id'].nunique():,}")
print(f"Payment types:\n{payments['payment_type'].value_counts()}")


Orders total:               99,441
Delivered orders:           96,476
Unique customers:           99,441

Payment rows:               103,886
Unique orders in payments:  99,440
Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


### Aggregate payments per order

- An order can be paid in multiple ways (e.g. voucher + credit card).
- Strategy: keep the dominant payment type (highest value), sum installments and total payment value across all payment rows for that order.

In [5]:
payments_agg = (
    payments
    .sort_values("payment_value", ascending=False)          # dominant type = highest value row
    .groupby("order_id")
    .agg(
        payment_type        = ("payment_type", "first"),    # dominant type
        payment_installments= ("payment_installments", "sum"),
        total_payment_value = ("payment_value", "sum"),
        n_payment_methods   = ("payment_type", "nunique")   # flag for split payments
    )
    .reset_index()
)
 
print(f"\nAggregated payment rows: {len(payments_agg):,}")
print(f"Split-payment orders:    {(payments_agg['n_payment_methods'] > 1).sum():,}")


Aggregated payment rows: 99,440
Split-payment orders:    2,246


### Aggregate items per order

In [6]:
# Translate product categories to English first
products_en = products.merge(category_translation, on="product_category_name", how="left")
 
items_agg = (
    items
    .merge(products_en[["product_id", "product_category_name_english"]], on="product_id", how="left")
    .groupby("order_id")
    .agg(
        total_price         = ("price", "sum"),
        total_freight       = ("freight_value", "sum"),
        n_items             = ("order_item_id", "count"),
        product_category    = ("product_category_name_english", "first")  # primary category
    )
    .reset_index()
)
 
items_agg["total_order_value"] = items_agg["total_price"] + items_agg["total_freight"]
 
print(f"\nAggregated item rows: {len(items_agg):,}")


Aggregated item rows: 98,666


### Keep one review per order (most recent if duplicates exist)

In [7]:
reviews_clean = (
    reviews
    .sort_values("review_creation_date", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "review_score", "review_creation_date"]]
)
 
print(f"\nReview rows (deduplicated): {len(reviews_clean):,}")


Review rows (deduplicated): 98,673


### Keep only delivered orders

In [8]:
# Delivery delta is only meaningful for orders that actually arrived
orders_delivered = orders[orders["order_status"] == "delivered"].copy()
print(f"\nDelivered orders: {len(orders_delivered):,}")


Delivered orders: 96,478


### Master join

In [9]:
df = (
    orders_delivered
    .merge(customers,     on="customer_id",  how="left")
    .merge(payments_agg,  on="order_id",     how="left")
    .merge(reviews_clean, on="order_id",     how="left")
    .merge(items_agg,     on="order_id",     how="left")
)
 
print(f"\nMaster dataframe shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Master dataframe shape: (96478, 23)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_type', 'payment_installments', 'total_payment_value', 'n_payment_methods', 'review_score', 'review_creation_date', 'total_price', 'total_freight', 'n_items', 'product_category', 'total_order_value']


### Check join quality

In [10]:
join_quality = pd.DataFrame({
    "column": df.columns,
    "null_count": df.isnull().sum().values,
    "null_pct": (df.isnull().sum().values / len(df) * 100).round(2)
}).query("null_count > 0").sort_values("null_pct", ascending=False)
 
print("\nColumns with nulls after join:")
print(join_quality.to_string(index=False))


Columns with nulls after join:
                       column  null_count  null_pct
             product_category        1351      1.40
                 review_score         646      0.67
         review_creation_date         646      0.67
            order_approved_at          14      0.01
order_delivered_customer_date           8      0.01
 order_delivered_carrier_date           2      0.00
          total_payment_value           1      0.00
         payment_installments           1      0.00
                 payment_type           1      0.00
            n_payment_methods           1      0.00


## **Feature Engineering**

### Delivery features

In [11]:
df["delivery_delta"] = (
    df["order_estimated_delivery_date"] - df["order_delivered_customer_date"]
).dt.days
# Positive = delivered before estimate (early), negative = late
 
df["delivery_speed_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days
 
df["is_late"] = (df["delivery_delta"] < 0).astype(int)
 
print("Delivery features:")
print(f"  Avg delivery delta (days): {df['delivery_delta'].mean():.1f}")
print(f"  Late orders: {df['is_late'].sum():,} ({df['is_late'].mean()*100:.1f}%)")

Delivery features:
  Avg delivery delta (days): 10.9
  Late orders: 7,826 (8.1%)


### Customer lifetime orders

In [12]:
# Note: customer_id is unique per order in Olist (same person = new customer_id)
# Use customer_unique_id from the customers table to count real repeat buyers
customer_order_counts = (
    df.groupby("customer_unique_id")["order_id"]
    .count()
    .reset_index()
    .rename(columns={"order_id": "customer_lifetime_orders"})
)
 
df = df.merge(customer_order_counts, on="customer_unique_id", how="left")
print(f"\nRepeat buyers (>1 order): {(df['customer_lifetime_orders'] > 1).sum():,}")


Repeat buyers (>1 order): 5,921


### One-hot encode payment type

In [13]:
payment_dummies = pd.get_dummies(df["payment_type"], prefix="payment")
df = pd.concat([df, payment_dummies], axis=1)
 
print(f"\nPayment dummy columns: {[c for c in df.columns if c.startswith('payment_')]}")


Payment dummy columns: ['payment_type', 'payment_installments', 'payment_boleto', 'payment_credit_card', 'payment_debit_card', 'payment_voucher']


### Binary satisfaction target

In [14]:
# Drop rows with no review score (orders without a review)
df = df.dropna(subset=["review_score"])
 
df["high_satisfaction"] = (df["review_score"] >= 4).astype(int)
 
print(f"\nSatisfaction split:")
print(df["high_satisfaction"].value_counts(normalize=True).round(3))


Satisfaction split:
high_satisfaction
1    0.789
0    0.211
Name: proportion, dtype: float64


### Final clean dataset

In [15]:
# Drop rows where key ML features are missing
ML_FEATURES = [
    "payment_installments", "total_price", "total_freight",
    "delivery_delta", "delivery_speed_days", "is_late",
    "total_order_value", "customer_lifetime_orders"
]
 
df_clean = df.dropna(subset=ML_FEATURES).copy()
 
print(f"\nFinal clean dataset: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
print(f"Dropped {len(df) - len(df_clean):,} rows with missing ML features")
 
# Quick preview
df_clean[[
    "order_id", "customer_state", "payment_type",
    "payment_installments", "total_order_value",
    "delivery_delta", "is_late", "review_score", "high_satisfaction"
]].head(5)


Final clean dataset: 95,823 rows × 32 columns
Dropped 9 rows with missing ML features


,order_id,customer_state,payment_type,payment_installments,total_order_value,delivery_delta,is_late,review_score,high_satisfaction
0,e481f51cbdc54678b7cc49136f2d6af7,SP,voucher,3.0,38.71,7.0,0,4.0,1
1,53cdb2fc8bc7dce0b6741e2150273451,BA,boleto,1.0,141.46,5.0,0,4.0,1
2,47770eb9100c2d0c44946d9cf07ec65d,GO,credit_card,3.0,179.12,17.0,0,5.0,1
3,949d5b44dbf5de918fe9c16f97b45f8a,RN,credit_card,1.0,72.20,12.0,0,5.0,1
4,ad21c59c0840e6cb83a9ceb5573f8159,SP,credit_card,1.0,28.62,9.0,0,5.0,1


## **EDA**

- dashboard.py  —  run with:  python dashboard.py
- then open http://127.0.0.1:8050 in your browser

In [16]:
# =============================================================================
# CONSTANTS & CONFIGURATION
# =============================================================================

COLOR_MAP = {
    "Credit card": "#185FA5", "Boleto": "#D85A30",
    "Voucher": "#1D9E75", "Debit card": "#7F77DD"
}

PAYMENT_LABELS = {
    "credit_card": "Credit card", "boleto": "Boleto",
    "voucher": "Voucher", "debit_card": "Debit card"
}

CHART_LAYOUT = dict(
    plot_bgcolor="white", paper_bgcolor="white",
    font_family="sans-serif", font_size=12,
    margin=dict(l=10, r=10, t=40, b=10),
)

BRAZIL_GEOJSON = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"

# =============================================================================
# DATA LOADING
# =============================================================================

def load_and_clean_data():
    path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
    DATA_DIR = Path(path)

    orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv",
                         parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
    payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
    reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
    customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
    items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
    products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
    cat_transl = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

    payments_agg = (
        payments.sort_values("payment_value", ascending=False)
        .groupby("order_id")
        .agg(payment_type=("payment_type","first"),
             payment_installments=("payment_installments","sum"),
             total_payment_value=("payment_value","sum"))
        .reset_index()
    )

    products_en = products.merge(cat_transl, on="product_category_name", how="left")
    items_agg = (
        items.merge(products_en[["product_id","product_category_name_english"]], on="product_id", how="left")
        .groupby("order_id")
        .agg(total_price=("price","sum"),
             total_freight=("freight_value","sum"),
             product_category=("product_category_name_english","first"))
        .reset_index()
    )
    items_agg["total_order_value"] = items_agg["total_price"] + items_agg["total_freight"]
    reviews_clean = reviews.sort_values("review_creation_date", ascending=False).drop_duplicates(subset="order_id", keep="first")[["order_id","review_score"]]

    df = (
        orders[orders["order_status"] == "delivered"]
        .merge(customers, on="customer_id", how="left")
        .merge(payments_agg, on="order_id", how="left")
        .merge(reviews_clean, on="order_id", how="left")
        .merge(items_agg, on="order_id", how="left")
    )

    df["delivery_delta"] = (df["order_estimated_delivery_date"] - df["order_delivered_customer_date"]).dt.days
    df["delivery_speed_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
    df["is_late"] = (df["delivery_delta"] < 0).astype(int)
    df["payment_label"] = df["payment_type"].map(PAYMENT_LABELS).fillna("Other")
    df["year"] = df["order_purchase_timestamp"].dt.year
    df["year_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
    
    # Customer Lifetime orders
    counts = df.groupby("customer_unique_id")["order_id"].count().reset_index().rename(columns={"order_id": "customer_lifetime_orders"})
    df = df.merge(counts, on="customer_unique_id", how="left")

    return df.dropna(subset=["payment_installments", "total_order_value", "review_score"])

df_clean = load_and_clean_data()
ALL_STATES = sorted(df_clean["customer_state"].unique())
ALL_YEARS = sorted(df_clean["year"].unique().astype(int))
ALL_PAYMENT_TYPES = [p for p in ["Credit card","Boleto","Voucher","Debit card"] if p in df_clean["payment_label"].unique()]

# =============================================================================
# UI COMPONENTS
# =============================================================================

def card(label, value, sub=""):
    return html.Div(style={"background":"white","borderRadius":"8px","padding":"15px","border":"0.5px solid #e8e8e8","flex":"1"}, children=[
        html.Div(label, style={"fontSize":"11px","color":"#888","textTransform":"uppercase"}),
        html.Div(value, style={"fontSize":"20px","fontWeight":"bold"}),
        html.Div(sub, style={"fontSize":"10px","color":"#aaa"})
    ])

def chart_card(figure, span_full=False):
    style = {"background":"white","borderRadius":"8px","border":"0.5px solid #e8e8e8","padding":"10px","marginBottom":"15px","width":"100%","boxSizing":"border-box"}
    if span_full: style["gridColumn"] = "1 / -1"
    return html.Div(style=style, children=[dcc.Graph(figure=figure, config={"displayModeBar":False, "responsive":True})])

# =============================================================================
# LAYOUT
# =============================================================================

app = dash.Dash(__name__, title="Olist Pro")

app.layout = html.Div(style={"fontFamily":"sans-serif","background":"#f4f4f2","minHeight":"100vh","paddingBottom":"30px"}, children=[
    
    # Header & Responsive Filters
    html.Div(style={"background":"white","padding":"15px 25px","display":"flex","flexWrap":"wrap","gap":"20px","alignItems":"center","borderBottom":"1px solid #ddd"}, children=[
        html.H3("Olist BI Dashboard", style={"margin":"0","fontSize":"18px","flex":"1"}),
        
        html.Div([html.Div("States", style={"fontSize":"10px"}), dcc.Dropdown(id="f-state", options=[{"label": s, "value": s} for s in ALL_STATES], multi=True, style={"minWidth":"200px"})]),
        html.Div([html.Div("Payment", style={"fontSize":"10px"}), dcc.Checklist(id="f-pay", options=[{"label": f" {p}", "value": p} for p in ALL_PAYMENT_TYPES], value=ALL_PAYMENT_TYPES, inline=True)]),
        html.Div([html.Div("Year", style={"fontSize":"10px"}), dcc.Checklist(id="f-year", options=[{"label": f" {y}", "value": y} for y in ALL_YEARS], value=ALL_YEARS, inline=True)]),
    ]),

    # KPI Row
    html.Div(id="kpi-row", style={"display":"flex","gap":"15px","padding":"20px 25px"}),

    dcc.Tabs(id="tabs", value="tab-overview", style={"padding":"0 25px"}, children=[
        dcc.Tab(label="Overview", value="tab-overview"),
        dcc.Tab(label="Payment Behavior", value="tab-payment"),
        dcc.Tab(label="Delivery & Satisfaction", value="tab-delivery"),
        dcc.Tab(label="Trends", value="tab-trends"),
    ]),

    # The Grid
    html.Div(id="tab-content", style={"display":"grid","gridTemplateColumns":"1fr 1fr","gap":"15px","padding":"20px 25px"})
])

# =============================================================================
# CALLBACK
# =============================================================================

@callback(
    Output("kpi-row", "children"),
    Output("tab-content", "children"),
    Input("f-state", "value"), Input("f-pay", "value"), Input("f-year", "value"), Input("tabs", "value")
)
def update_dashboard(selected_states, payment_types, years, tab):
    # 1. Filter
    d = df_clean.copy()
    if selected_states: d = d[d["customer_state"].isin(selected_states)]
    if payment_types: d = d[d["payment_label"].isin(payment_types)]
    if years: d = d[d["year"].isin(years)]

    if d.empty: return [], [html.Div("No data matches filters.", style={"gridColumn":"1/-1","textAlign":"center"})]

    # 2. KPIs
    kpis = [
        card("Orders", f"{len(d):,}"),
        card("Avg Score", f"{d['review_score'].mean():.2f}"),
        card("Avg Delivery Delay", f"{d['delivery_delta'].mean():.1f} days"),
        card("Late Orders", f"{d['is_late'].mean()*100:.1f}%")
    ]

    lo = {k: v for k, v in CHART_LAYOUT.items()}

    if tab == "tab-overview":
        # median installments by category
        cat = d.dropna(subset=["product_category"]).groupby("product_category")["payment_installments"].median().reset_index().sort_values("payment_installments", ascending=False).head(10)
        figA = px.bar(cat, x="payment_installments", y="product_category", orientation="h", title="Top 10 Categories by Median Installments", color_discrete_sequence=["#185FA5"])
        figA.update_layout(**lo, height=350, yaxis={'categoryorder':'total ascending'})

        # Payment share by state
        pbs = d.groupby(["customer_state","payment_label"]).size().reset_index(name="c")
        pbs["share"] = (pbs["c"] / pbs.groupby("customer_state")["c"].transform("sum") * 100)
        figB = px.bar(pbs, x="customer_state", y="share", color="payment_label", color_discrete_map=COLOR_MAP, title="Payment Method Share by State")
        figB.update_layout(**lo, height=350, barmode="stack", xaxis_tickangle=-45)
        
        return kpis, [chart_card(figA, span_full=True), chart_card(figB, span_full=True)]

    elif tab == "tab-payment":
        # Histogram
        figA = px.histogram(d, x="payment_installments", title="Installment Distribution", color_discrete_sequence=["#185FA5"])
        figA.update_layout(**lo, height=300)

        # Correlation (FULL ITEMS RESTORED)
        corr_cols = ["payment_installments", "total_order_value", "total_freight", "delivery_delta", "delivery_speed_days", "is_late", "review_score", "customer_lifetime_orders"]
        cm = d[corr_cols].corr().round(2)
        mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
        cm_masked = cm.where(~mask)
        figB = px.imshow(cm_masked, text_auto=True, color_continuous_scale="RdBu", zmin=-1, zmax=1, title="Correlation Matrix (Lower Triangle)")
        figB.update_layout(**lo, height=450)

        return kpis, [chart_card(figA, span_full=True), chart_card(figB, span_full=True)]

    elif tab == "tab-delivery":
        # Brazil Map
        map_all = df_clean.groupby("customer_state").agg(t=("order_id","count"), l=("is_late","sum")).reset_index()
        map_all["late_rate"] = (map_all["l"] / map_all["t"]) * 100
        if selected_states:
            map_all["display"] = map_all.apply(lambda x: x["late_rate"] if x["customer_state"] in selected_states else None, axis=1)
        else:
            map_all["display"] = map_all["late_rate"]

        fig_map = px.choropleth(map_all, geojson=BRAZIL_GEOJSON, locations="customer_state", featureidkey="properties.sigla",
                                color="display", color_continuous_scale="Reds", title="Late Delivery % by State")
        fig_map.update_geos(fitbounds="locations", visible=False)
        fig_map.update_layout(**lo, height=450)

        # Violin
        figV = go.Figure()
        for pt, color in COLOR_MAP.items():
            subset = d[d["payment_label"] == pt]
            if not subset.empty:
                figV.add_trace(go.Violin(x=subset["payment_label"], y=subset["review_score"], name=pt, line_color=color, points=False, box_visible=True))
        figV.update_layout(**lo, title="Review Score by Payment Type", height=350, showlegend=False)

        # Box
        figBox = px.box(d, x="payment_label", y="delivery_delta", color="payment_label", color_discrete_map=COLOR_MAP, title="Delivery Delta (vs Estimate)")
        figBox.update_layout(**lo, height=350, showlegend=False)

        # Late Bar
        late_st = d.groupby("customer_state")["is_late"].mean().reset_index().sort_values("is_late", ascending=False)
        figBar = px.bar(late_st, x="customer_state", y="is_late", title="Late Rate by Selected States", color="is_late", color_continuous_scale="Reds")
        figBar.update_layout(**lo, height=350, coloraxis_showscale=False)

        return kpis, [chart_card(fig_map, span_full=True), chart_card(figV), chart_card(figBox), chart_card(figBar, span_full=True)]

    else: # Trends
        trend = d.groupby(["year_month","payment_label"])["review_score"].mean().reset_index()
        figA = px.line(trend, x="year_month", y="review_score", color="payment_label", color_discrete_map=COLOR_MAP, title="Avg Review Score Trend")
        figA.update_layout(**lo, height=400, xaxis_tickangle=-45)
        return kpis, [chart_card(figA, span_full=True)]

if __name__ == "__main__":
    app.run(debug=True, port=8050)